# Analyze HW3 Prompt Experiment Results

This notebook loads experiment scores from `outputs/experiment_results.csv` and performs:

- Descriptive statistics by prompt (A/B/C)
- One-way ANOVA on `overall_score`
- A boxplot comparing prompt score distributions

Post-hoc pairwise tests are intentionally omitted; this workflow reports ANOVA only.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    import pingouin as pg
except ImportError as exc:
    raise ImportError("Install pingouin first: pip install pingouin") from exc

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


In [ ]:
results_path = Path("outputs/experiment_results.csv")
if not results_path.exists():
    raise FileNotFoundError(
        f"Results file not found at {results_path}. Run run_experiment.py first or update results_path."
    )

scores = pd.read_csv(results_path)
scores["prompt_id"] = scores["prompt_id"].astype(str)

print(f"Loaded {len(scores)} rows from {results_path}")
display(scores.head())


In [ ]:
score_cols = [
    "overall_score",
    "numeric_fidelity",
    "legal_claim_risk",
    "source_attribution_coverage",
    "missingness_disclosure",
    "public_utility_rating",
    "legal_claim_safety",
    "coverage_required_public_questions",
]

descriptive_stats = (
    scores.groupby("prompt_id")[score_cols]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .round(3)
)

print("Descriptive statistics by prompt:")
display(descriptive_stats)

overall_summary = (
    scores.groupby("prompt_id")["overall_score"]
    .agg(["count", "mean", "std"])
    .sort_values("mean", ascending=False)
    .round(3)
)

print("Overall score summary by prompt:")
display(overall_summary)


In [ ]:
variance_test = pg.homoscedasticity(data=scores, dv="overall_score", group="prompt_id", method="bartlett")
equal_variance = bool(variance_test["equal_var"].iloc[0])
variance_p = float(variance_test["pval"].iloc[0])

display(variance_test)
print(f"Equal variance assumption: {equal_variance} (Bartlett p={variance_p:.4f})")

if equal_variance:
    anova_result = pg.anova(data=scores, dv="overall_score", between="prompt_id", detailed=True)
    anova_name = "One-way ANOVA"
else:
    anova_result = pg.welch_anova(data=scores, dv="overall_score", between="prompt_id")
    anova_name = "Welch ANOVA"

display(anova_result)

p_value = float(anova_result["p-unc"].iloc[0])
print(f"{anova_name} p-value: {p_value:.6f}")
if p_value < 0.05:
    print("Result: reject H0. At least one prompt mean differs.")
else:
    print("Result: fail to reject H0. No statistically significant mean difference found.")


In [ ]:
print("Post-hoc pairwise tests are intentionally omitted in this notebook.")
print("Use ANOVA output above as the sole inferential test for this assignment.")


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.boxplot(data=scores, x="prompt_id", y="overall_score", order=["A", "B", "C"])
sns.stripplot(
    data=scores,
    x="prompt_id",
    y="overall_score",
    order=["A", "B", "C"],
    color="black",
    alpha=0.45,
    size=3,
    jitter=0.2,
)
ax.set_title("Overall Validation Score by Prompt")
ax.set_xlabel("Prompt")
ax.set_ylabel("Overall Score (0-100)")
plt.tight_layout()
plt.show()
